In [2]:
import json, math, torch, torch.nn as nn, torch.nn.functional as F
from dataclasses import dataclass
import sentencepiece as spm

DEVICE = "cuda"
torch.manual_seed(0)

sp = spm.SentencePieceProcessor(model_file="data/spm.model")
PAD, UNK, BOS, EOS = 0, 1, 2, 3
LANG_TAG = {l: sp.piece_to_id(f"<2{l}>") for l in ["hi", "zh", "yo"]}
print("vocab:", sp.get_piece_size(), "| lang tags:", LANG_TAG)

@dataclass
class Config:
    vocab_size: int = sp.get_piece_size()
    d_model: int = 512
    n_head: int = 8
    d_mlp: int = 2048
    n_enc: int = 3
    n_dec: int = 3
    max_len: int = 256          # position table size (sentences are capped at 128)
    dropout: float = 0.1
    label_smoothing: float = 0.1

cfg = Config()

vocab: 24000 | lang tags: {'hi': 4, 'zh': 5, 'yo': 6}


In [3]:
class Attention(nn.Module):
    """Self-attention if mem is None, cross-attention otherwise."""
    def __init__(self, c):
        super().__init__()
        self.h, self.p = c.n_head, c.dropout
        self.q = nn.Linear(c.d_model, c.d_model)
        self.kv = nn.Linear(c.d_model, 2 * c.d_model)
        self.proj = nn.Linear(c.d_model, c.d_model)

    def forward(self, x, mem=None, mask=None, causal=False):
        B, T, C = x.shape
        src = x if mem is None else mem
        S = src.size(1)
        q = self.q(x).view(B, T, self.h, C // self.h).transpose(1, 2)
        k, v = self.kv(src).split(C, dim=2)
        k = k.view(B, S, self.h, C // self.h).transpose(1, 2)
        v = v.view(B, S, self.h, C // self.h).transpose(1, 2)
        y = F.scaled_dot_product_attention(
            q, k, v, attn_mask=mask, is_causal=causal,
            dropout_p=self.p if self.training else 0.0)
        return self.proj(y.transpose(1, 2).contiguous().view(B, T, C))

class MLP(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.fc = nn.Linear(c.d_model, c.d_mlp)
        self.act = nn.GELU()                    # hook here -> the "neurons"
        self.proj = nn.Linear(c.d_mlp, c.d_model)
        self.drop = nn.Dropout(c.dropout)

    def forward(self, x):
        return self.drop(self.proj(self.act(self.fc(x))))

class EncoderBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.ln1, self.attn = nn.LayerNorm(c.d_model), Attention(c)
        self.ln2, self.mlp = nn.LayerNorm(c.d_model), MLP(c)
        self.drop = nn.Dropout(c.dropout)

    def forward(self, x, src_mask):
        x = x + self.drop(self.attn(self.ln1(x), mask=src_mask))
        return x + self.mlp(self.ln2(x))

class DecoderBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.ln1, self.self_attn = nn.LayerNorm(c.d_model), Attention(c)
        self.ln2, self.cross_attn = nn.LayerNorm(c.d_model), Attention(c)
        self.ln3, self.mlp = nn.LayerNorm(c.d_model), MLP(c)
        self.drop = nn.Dropout(c.dropout)

    def forward(self, x, mem, src_mask):
        x = x + self.drop(self.self_attn(self.ln1(x), causal=True))
        x = x + self.drop(self.cross_attn(self.ln2(x), mem=mem, mask=src_mask))
        return x + self.mlp(self.ln3(x))

class Seq2Seq(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.cfg = c
        self.tok_emb = nn.Embedding(c.vocab_size, c.d_model)
        self.enc_pos = nn.Embedding(c.max_len, c.d_model)
        self.dec_pos = nn.Embedding(c.max_len, c.d_model)
        self.enc_blocks = nn.ModuleList([EncoderBlock(c) for _ in range(c.n_enc)])
        self.dec_blocks = nn.ModuleList([DecoderBlock(c) for _ in range(c.n_dec)])
        self.enc_ln, self.dec_ln = nn.LayerNorm(c.d_model), nn.LayerNorm(c.d_model)
        self.drop = nn.Dropout(c.dropout)
        self.lm_head = nn.Linear(c.d_model, c.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight          # one shared table
        self.apply(self._init)
        for name, p in self.named_parameters():            # GPT-2 style residual scaling
            if name.endswith("proj.weight"):
                nn.init.normal_(p, std=0.02 / math.sqrt(2 * (c.n_enc + c.n_dec)))

    def _init(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def encode(self, src):
        S = src.size(1)
        assert S <= self.cfg.max_len
        src_mask = (src != PAD)[:, None, None, :]          # (B,1,1,S): True = may attend
        x = self.drop(self.tok_emb(src) + self.enc_pos(torch.arange(S, device=src.device)))
        for b in self.enc_blocks:
            x = b(x, src_mask)
        return self.enc_ln(x), src_mask

    def decode(self, tgt_in, mem, src_mask):
        T = tgt_in.size(1)
        assert T <= self.cfg.max_len
        x = self.drop(self.tok_emb(tgt_in) + self.dec_pos(torch.arange(T, device=tgt_in.device)))
        for b in self.dec_blocks:
            x = b(x, mem, src_mask)
        return self.lm_head(self.dec_ln(x))

    def forward(self, src, tgt_in, tgt_out=None):
        mem, src_mask = self.encode(src)
        logits = self.decode(tgt_in, mem, src_mask)
        loss = None
        if tgt_out is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), tgt_out.view(-1),
                                   ignore_index=PAD, label_smoothing=self.cfg.label_smoothing)
        return logits, loss

## turning sentence pairs into tensors

In [4]:
MAX_TOK = 128

def pad_batch(seqs):
    L = max(len(s) for s in seqs)
    return torch.tensor([s + [PAD] * (L - len(s)) for s in seqs], device=DEVICE)

def make_batch(pairs, lang="hi"):
    """pairs: list of {"en": ..., "tgt": ...}.
    src     = English tokens + </s>
    tgt_in  = <2xx> + target tokens      (what the decoder reads)
    tgt_out = target tokens + </s>       (what it must predict)"""
    src, tgt_in, tgt_out = [], [], []
    for p in pairs:
        s = sp.encode(p["en"])[:MAX_TOK - 1] + [EOS]
        t = sp.encode(p["tgt"])[:MAX_TOK - 1]
        src.append(s)
        tgt_in.append([LANG_TAG[lang]] + t)
        tgt_out.append(t + [EOS])
    return pad_batch(src), pad_batch(tgt_in), pad_batch(tgt_out)

## Cell 4: sanity checks

In [5]:
model = Seq2Seq(cfg).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"params: {n_params/1e6:.1f}M")

val = [json.loads(l) for l in open("data/en-hi/val.jsonl")][:32]
src, tgt_in, tgt_out = make_batch(val)
print("shapes:", src.shape, tgt_in.shape, tgt_out.shape)

# 1) Initial loss should be about ln(vocab size)
model.eval()
with torch.no_grad():
    _, loss = model(src, tgt_in, tgt_out)
print(f"initial loss {loss.item():.2f} (expect ~{math.log(cfg.vocab_size):.2f})")

# 2) Hooks: every MLP layer should give (batch, length, 2048)
acts = {}
hooks = []
for i, b in enumerate(model.enc_blocks):
    hooks.append(b.mlp.act.register_forward_hook(lambda m, inp, out, i=i: acts.__setitem__(f"enc{i}", out.shape)))
for i, b in enumerate(model.dec_blocks):
    hooks.append(b.mlp.act.register_forward_hook(lambda m, inp, out, i=i: acts.__setitem__(f"dec{i}", out.shape)))
with torch.no_grad():
    model(src, tgt_in, tgt_out)
for h in hooks:
    h.remove()
print("MLP activation shapes:", acts)



params: 34.6M
shapes: torch.Size([32, 49]) torch.Size([32, 48]) torch.Size([32, 48])
initial loss 10.12 (expect ~10.09)
MLP activation shapes: {'enc0': torch.Size([32, 49, 2048]), 'enc1': torch.Size([32, 49, 2048]), 'enc2': torch.Size([32, 49, 2048]), 'dec0': torch.Size([32, 48, 2048]), 'dec1': torch.Size([32, 48, 2048]), 'dec2': torch.Size([32, 48, 2048])}


## Should Perfectly Memorize a small batch and translate properly

In [6]:
# 3) Overfit test: the model should memorise 32 pairs quickly. If loss won't drop, something is wrong.
model.train()
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
for step in range(300):
    _, loss = model(src, tgt_in, tgt_out)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 50 == 0:
        print(f"step {step}: loss {loss.item():.3f}")

step 0: loss 10.136
step 50: loss 5.432
step 100: loss 2.235
step 150: loss 1.508
step 200: loss 1.453
step 250: loss 1.402


In [7]:
@torch.no_grad()
def translate(model, sents, lang="hi", max_new=MAX_TOK, bs=64):
    model.eval()
    out = []
    for i in range(0, len(sents), bs):
        src = pad_batch([sp.encode(s)[:MAX_TOK - 1] + [EOS] for s in sents[i:i + bs]])
        mem, src_mask = model.encode(src)
        ys = torch.full((src.size(0), 1), LANG_TAG[lang], device=DEVICE)
        done = torch.zeros(src.size(0), dtype=torch.bool, device=DEVICE)
        for _ in range(max_new):
            nxt = model.decode(ys, mem, src_mask)[:, -1].argmax(-1)
            nxt[done] = PAD
            ys = torch.cat([ys, nxt[:, None]], dim=1)
            done |= nxt == EOS
            if done.all():
                break
        for row in ys[:, 1:].tolist():
            row = row[:row.index(EOS)] if EOS in row else row
            out.append(sp.decode([t for t in row if t != PAD]))
    return out

# After the overfit test, it should reproduce memorised training sentences
for p, hyp in zip(val[:3], translate(model, [p["en"] for p in val[:3]])):
    print("EN :", p["en"]); print("REF:", p["tgt"]); print("OUT:", hyp); print()

EN : Show names
REF: नाम दिखाएं
OUT: नाम दिखाएं

EN : slicker
REF: बकव्रती
OUT: बकव्रती

EN : Jorhat
REF: जोरहाट
OUT: जोरहाट



## Drop instances with v few english words

In [8]:
train = [json.loads(l) for l in open("data/en-hi/train.jsonl")]
short = sum(len(p["en"].split()) <= 2 for p in train)
print(f"{short:,} of {len(train):,} pairs ({100*short/len(train):.0f}%) have 2 or fewer English words")

295,381 of 1,288,397 pairs (23%) have 2 or fewer English words


In [10]:
print(1)

1


In [11]:
import json

for split in ["train", "val"]:
    path = f"data/en-hi/{split}.jsonl"
    rows = [json.loads(l) for l in open(path, encoding="utf-8")]
    kept = [r for r in rows if len(r["en"].split()) >= 3]
    with open(path, "w", encoding="utf-8") as f:
        for r in kept:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"{split}: {len(rows):,} -> {len(kept):,}")

train: 1,288,397 -> 993,016
val: 5,000 -> 3,872


## Training Setups

In [12]:
pip install sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 886.5 kB/s  0:00:05 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.8/807.8 kB 797.1 kB/s  0:00:01eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [sacrebleu]/6 [colorama]
Note: you may need to restart the kernel to use updated packages.


In [13]:
import json, os, random, time, math
from dataclasses import dataclass, asdict
from sacrebleu.metrics import CHRF

torch.backends.cuda.matmul.allow_tf32 = True     # faster matmuls on the 3090, no accuracy cost here

@dataclass
class TrainConfig:
    run_name: str = "en-hi_s0"
    lang: str = "hi"
    seed: int = 0
    max_tokens: int = 16000      # tokens per batch (counted with padding, per side)
    lr: float = 5e-4
    warmup_steps: int = 4000     # linear warmup, then constant (same rule we'll use per phase later)
    weight_decay: float = 0.01
    grad_clip: float = 1.0
    eval_every: int = 1000       # validation loss every N steps
    chrf_every: int = 5000       # chrF on FLORES devtest every N steps (slower: full translation)
    min_steps: int = 10000       # no plateau check before this
    max_steps: int = 60000       # hard cap
    plateau_window: int = 5      # compare the mean of the last 5 evals with the 5 before
    plateau_tol: float = 0.002   # stop if relative improvement < 0.2%

T = TrainConfig()
RUN_DIR = f"runs/{T.run_name}"
os.makedirs(RUN_DIR, exist_ok=True)
random.seed(T.seed); torch.manual_seed(T.seed)

def load_pairs(path, lang):
    """Read a jsonl of {"en","tgt"} pairs and tokenize once.
    Each example becomes (src, tgt_in, tgt_out) as lists of token ids:
      src     = English tokens + </s>
      tgt_in  = <2xx> + target tokens   (decoder input)
      tgt_out = target tokens + </s>    (what the decoder must predict)"""
    rows = [json.loads(l) for l in open(path, encoding="utf-8")]
    src_ids = sp.encode([r["en"] for r in rows])
    tgt_ids = sp.encode([r["tgt"] for r in rows])
    data = []
    for s, t in zip(src_ids, tgt_ids):
        s = s[:MAX_TOK - 1] + [EOS]
        t = t[:MAX_TOK - 1]
        data.append((s, [LANG_TAG[lang]] + t, t + [EOS]))
    return data

t0 = time.time()
train_data = load_pairs("data/en-hi/train.jsonl", T.lang)
val_data = load_pairs("data/en-hi/val.jsonl", T.lang)
print(f"tokenized {len(train_data):,} train / {len(val_data):,} val pairs in {time.time()-t0:.0f}s")

# FLORES devtest for chrF (dev is kept for probing, never used here)
flores_src = [json.loads(l)["text"] for l in open("data/flores/devtest.en.jsonl", encoding="utf-8")]
flores_ref = [json.loads(l)["text"] for l in open(f"data/flores/devtest.{T.lang}.jsonl", encoding="utf-8")]
print(f"FLORES devtest: {len(flores_src)} sentences")

tokenized 993,016 train / 3,872 val pairs in 43s
FLORES devtest: 1012 sentences


In [14]:
def make_batches(data, max_tokens, seed=0, shuffle=True):
    """Group examples of similar length so batches waste little space on padding.
    1. Shuffle all indices (so each epoch differs).
    2. Within chunks of 100k examples, sort by length.
    3. Fill batches until (longest sentence x number of sentences) would exceed max_tokens.
    4. Shuffle the order of the batches."""
    rng = random.Random(seed)
    idx = list(range(len(data)))
    if shuffle:
        rng.shuffle(idx)
    length = lambda i: max(len(data[i][0]), len(data[i][1]))
    batches = []
    for c in range(0, len(idx), 100_000):
        chunk = sorted(idx[c:c + 100_000], key=length)
        cur, cur_max = [], 0
        for i in chunk:
            L = length(i)
            if cur and max(cur_max, L) * (len(cur) + 1) > max_tokens:
                batches.append(cur)
                cur, cur_max = [], 0
            cur.append(i)
            cur_max = max(cur_max, L)
        if cur:
            batches.append(cur)
    if shuffle:
        rng.shuffle(batches)
    return batches

def collate(data, batch_idx):
    """Turn a list of example indices into three padded tensors on the GPU."""
    src = pad_batch([data[i][0] for i in batch_idx])
    tgt_in = pad_batch([data[i][1] for i in batch_idx])
    tgt_out = pad_batch([data[i][2] for i in batch_idx])
    return src, tgt_in, tgt_out

b = make_batches(train_data, T.max_tokens)
print(f"{len(b):,} batches per epoch, ~{len(train_data)/len(b):.0f} pairs per batch")

1,697 batches per epoch, ~585 pairs per batch


In [15]:
chrf_metric = CHRF()

@torch.no_grad()
def val_loss(model, data):
    """Mean cross-entropy per target token on the validation set.
    Plain cross-entropy, WITHOUT label smoothing, so the number is a clean measure
    (training loss includes smoothing and sits higher)."""
    model.eval()
    total, count = 0.0, 0
    for bidx in make_batches(data, T.max_tokens, shuffle=False):
        src, tgt_in, tgt_out = collate(data, bidx)
        with torch.autocast("cuda", dtype=torch.bfloat16):
            logits, _ = model(src, tgt_in)
        loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), tgt_out.view(-1),
                               ignore_index=PAD, reduction="sum")
        total += loss.item()
        count += (tgt_out != PAD).sum().item()
    model.train()
    return total / count

def flores_chrf(model, n_show=2):
    """Greedy-translate FLORES devtest and score with chrF (0-100, higher is better).
    Also prints a few translations so you can judge them by eye."""
    with torch.autocast("cuda", dtype=torch.bfloat16):
        hyps = translate(model, flores_src, lang=T.lang)
    model.train()
    for k in range(n_show):
        print(f"    EN : {flores_src[k]}\n    REF: {flores_ref[k]}\n    OUT: {hyps[k]}")
    return chrf_metric.corpus_score(hyps, [flores_ref]).score

def lr_at(step):
    """Linear warmup from 0 to T.lr over warmup_steps, then constant."""
    return T.lr * min(1.0, (step + 1) / T.warmup_steps)

def plateaued(history):
    """True if the mean of the last `window` val losses improved by less than
    plateau_tol (relative) over the mean of the `window` before it."""
    w = T.plateau_window
    if len(history) < 2 * w:
        return False
    prev = sum(history[-2 * w:-w]) / w
    recent = sum(history[-w:]) / w
    return (prev - recent) / prev < T.plateau_tol

def save_ckpt(path, model, opt, step, best_val):
    """Save to a temp file, then rename, so a crash never leaves a half-written file."""
    tmp = path + ".tmp"
    torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "step": step,
                "best_val": best_val, "cfg": asdict(cfg), "train_cfg": asdict(T)}, tmp)
    os.replace(tmp, path)

## Training Loop

In [ ]:
model = Seq2Seq(cfg).to(DEVICE)                  # fresh weights
opt = torch.optim.AdamW(model.parameters(), lr=T.lr, betas=(0.9, 0.98), eps=1e-9,
                        weight_decay=T.weight_decay)
with open(f"{RUN_DIR}/config.json", "w") as f:
    json.dump({"model": asdict(cfg), "train": asdict(T)}, f, indent=2)
log = open(f"{RUN_DIR}/log.jsonl", "a")

step, epoch = 0, 0
best_val, val_hist = float("inf"), []
loss_sum, n_loss, tok_count = 0.0, 0, 0
t_start = t_last = time.time()
stop_reason = None

while stop_reason is None:
    for bidx in make_batches(train_data, T.max_tokens, seed=T.seed + epoch):
        src, tgt_in, tgt_out = collate(train_data, bidx)

        # --- one training step ---
        for g in opt.param_groups:
            g["lr"] = lr_at(step)
        with torch.autocast("cuda", dtype=torch.bfloat16):    # bf16 forward pass for speed
            _, loss = model(src, tgt_in, tgt_out)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), T.grad_clip)
        opt.step()
        step += 1

        loss_sum += loss.item(); n_loss += 1
        tok_count += (tgt_out != PAD).sum().item()

        # --- periodic evaluation ---
        if step % T.eval_every == 0:
            vl = val_loss(model, val_data)
            val_hist.append(vl)
            now = time.time()
            row = {"step": step, "epoch": epoch, "lr": lr_at(step),
                   "train_loss": loss_sum / n_loss,            # includes label smoothing
                   "val_loss": vl,                             # plain cross-entropy
                   "tgt_tok_per_s": tok_count / (now - t_last),
                   "minutes": (now - t_start) / 60}
            loss_sum, n_loss, tok_count, t_last = 0.0, 0, 0, now

            if step % T.chrf_every == 0:
                row["chrf"] = flores_chrf(model)

            if vl < best_val:                                  # keep the best model
                best_val = vl
                save_ckpt(f"{RUN_DIR}/best.pt", model, opt, step, best_val)
            save_ckpt(f"{RUN_DIR}/last.pt", model, opt, step, best_val)   # for resuming

            log.write(json.dumps(row) + "\n"); log.flush()
            print(f"step {step:6d} | ep {epoch} | train {row['train_loss']:.3f} | val {vl:.3f} | "
                  f"{row['tgt_tok_per_s']/1000:.0f}k tok/s | {row['minutes']:.1f} min"
                  + (f" | chrF {row['chrf']:.1f}" if "chrf" in row else ""))

            # --- stopping rules ---
            if step >= T.min_steps and plateaued(val_hist):
                stop_reason = "plateau"
            elif step >= T.max_steps:
                stop_reason = "max_steps"
            if stop_reason:
                break
    epoch += 1

# final score with the best checkpoint
model.load_state_dict(torch.load(f"{RUN_DIR}/best.pt")["model"])
final_chrf = flores_chrf(model, n_show=3)
print(f"\nstopped: {stop_reason} at step {step} | best val {best_val:.3f} | final chrF {final_chrf:.1f}")
log.write(json.dumps({"final": True, "stop_reason": stop_reason, "step": step,
                      "best_val": best_val, "chrf": final_chrf}) + "\n")
log.close()

step   1000 | ep 0 | train 7.074 | val 4.935 | 114k tok/s | 2.2 min
step   2000 | ep 1 | train 5.238 | val 3.885 | 104k tok/s | 4.5 min
step   3000 | ep 1 | train 4.582 | val 3.234 | 109k tok/s | 6.8 min
step   4000 | ep 2 | train 4.083 | val 2.745 | 113k tok/s | 8.9 min
    EN : "We now have 4-month-old mice that are non-diabetic that used to be diabetic," he added.
    REF: उन्होंने कहा “कि अब हमारे पास 4 महीने उम्र वाले चूहे हैं जिन्हें मधुमेह नहीं है जो मधुमेह के रोगी थे। ”
    OUT: "हम अब 4-महारा मिल है जो अहिंसा के लिए प्रयुक्त होता है"
    EN : Dr. Ehud Ur, professor of medicine at Dalhousie University in Halifax, Nova Scotia and chair of the clinical and scientific division of the Canadian Diabetes Association cautioned that the research is still in its early days.
    REF: डॉ. एहुड उर, नोवा स्कोटिया के हैलिफ़ैक्स में डलहौज़ी विश्वविद्यालय में चिकित्सा के प्रोफ़ेसर और कनाडाई डायबिटीज़ एसोसिएशन के नैदानिक ​​और वैज्ञानिक विभाग के अध्यक्ष ने आगाह किया कि शोध अभी भी अपने शुरुआती दि

In [1]:
print(1)

1
